# 02 — Leakage Audit & Temporal Sanity Checks

**Project:** Cinema Ticket Demand Forecasting (`T-48h`)  
**Objective:** Audit every engineered feature's exact availability timestamp, verify zero target/future leakage across the `T-48h` boundary, confirm train-only preprocessing, and run shuffled-time / feature-ablation sanity checks.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.config import load_config
from src.features import ALL_FEATURE_COLUMNS, CATEGORICAL_FEATURES, FEATURE_AVAILABILITY, NUMERIC_FEATURES, build_features
from src.split import split_chronological

cfg = load_config(PROJECT_ROOT / "config.yaml")
clean_df = pd.read_parquet(PROJECT_ROOT / cfg["paths"]["cleaned_data"])
feat_df = pd.read_parquet(PROJECT_ROOT / cfg["paths"]["features_data"])
print(f"Loaded {len(feat_df):,} rows and {len(ALL_FEATURE_COLUMNS)} features.")

Loaded 21,584 rows and 30 features.


## 1. Feature Availability Registry
Every model input must be genuinely known at or before `show_datetime - 48 hours`.

In [2]:
audit_table = pd.DataFrame([
    {"feature": col, "dtype": str(feat_df[col].dtype), "available_at": FEATURE_AVAILABILITY[col]}
    for col in ALL_FEATURE_COLUMNS
])
print(audit_table.to_string(index=False))

                  feature   dtype                                                       available_at
                     slot  object                                  Schedule creation (showtime slot)
                screen_id  object                                Schedule creation (assigned screen)
              screen_type  object                                  Schedule creation (screen format)
                    genre  object                                       Movie metadata (pre-release)
                 language  object                                       Movie metadata (pre-release)
                   rating  object                                       Movie metadata (pre-release)
                  weekday   int64                                Schedule creation (static calendar)
                    month   int64                                Schedule creation (static calendar)
               is_weekend   int64                                Schedule creation (static 

## 2. Perturbation Proof: Future & Within-Horizon Targets Have Zero Effect
We corrupt `occupancy_rate = 0.99` and `sold_out = 1` for every show at `>= target_dt - 48h` and assert all features at `target_dt` remain bit-for-bit identical.

In [3]:
sample_slice = clean_df.head(3000).copy()
probe_idx = 2000
probe_dt = pd.to_datetime(sample_slice.iloc[probe_idx]["show_datetime"])
cutoff_dt = probe_dt - pd.Timedelta(hours=48)

orig_row = build_features(sample_slice, horizon_hours=48).iloc[probe_idx][ALL_FEATURE_COLUMNS]

mutated = sample_slice.copy()
mask = pd.to_datetime(mutated["show_datetime"]) >= cutoff_dt
mutated.loc[mask, "occupancy_rate"] = 0.99
mutated.loc[mask, "sold_out"] = 1
mut_row = build_features(mutated, horizon_hours=48).iloc[probe_idx][ALL_FEATURE_COLUMNS]

pd.testing.assert_series_equal(orig_row, mut_row)
print(f"PASS: Perturbing {mask.sum()} shows >= {cutoff_dt} caused 0.0 difference at {probe_dt}.")

PASS: Perturbing 1081 shows >= 2024-02-18 11:00:00 caused 0.0 difference at 2024-02-20 11:00:00.


## 3. Chronological Split & Shuffled-Time / Ablation Sanity Check
Using train and validation splits only (test remains untouched), we verify:
- **Normal T-48h Pipeline:** Ridge trained on train, evaluated on validation.
- **Shuffled-Time Check:** Permuting target `occupancy_rate` breaks temporal alignment; validation MAE degrades sharply (proving the model relies on genuine signal, not hardcoded/constant leaks).
- **Remove-Suspicious-Feature Check:** Dropping `movie_effective_occ_prior` and `movie_occ_mean_7d` tests whether one feature dominates or whether calendar/slot/age features still provide strong signal.

In [4]:
train_df, val_df, test_df, dates = split_chronological(feat_df)
print(f"Train <= {dates.train_end_date} ({len(train_df):,}) | Val <= {dates.val_end_date} ({len(val_df):,}) | Test > {dates.val_end_date} ({len(test_df):,})")
assert train_df["show_datetime"].max() < val_df["show_datetime"].min() < test_df["show_datetime"].min()

def eval_ridge(tr, va, num_cols, cat_cols, shuffle_target=False):
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    pipe = Pipeline([("pre", pre), ("reg", Ridge(alpha=1.0))])
    y_tr = tr["occupancy_rate"].sample(frac=1.0, random_state=42).to_numpy() if shuffle_target else tr["occupancy_rate"].to_numpy()
    pipe.fit(tr[num_cols + cat_cols], y_tr)
    pred_occ = np.clip(pipe.predict(va[num_cols + cat_cols]), 0.0, 1.0)
    pred_tix = np.clip(pred_occ * va["capacity"].to_numpy(), 0.0, va["capacity"].to_numpy())
    return mean_absolute_error(va["tickets_sold"], pred_tix), mean_absolute_error(va["occupancy_rate"], pred_occ) * 100

mae_t_ok, mae_o_ok = eval_ridge(train_df, val_df, NUMERIC_FEATURES, CATEGORICAL_FEATURES, shuffle_target=False)
mae_t_shuf, mae_o_shuf = eval_ridge(train_df, val_df, NUMERIC_FEATURES, CATEGORICAL_FEATURES, shuffle_target=True)
lag_cols = {"movie_effective_occ_prior", "movie_occ_mean_7d", "movie_occ_mean_14d", "movie_occ_mean_all", "movie_slot_occ_mean", "movie_weekday_occ_mean"}
no_lag_num = [c for c in NUMERIC_FEATURES if c not in lag_cols]
mae_t_nolag, mae_o_nolag = eval_ridge(train_df, val_df, no_lag_num, CATEGORICAL_FEATURES, shuffle_target=False)

print(f"Normal Ridge (Val):           MAE = {mae_t_ok:.2f} tickets | {mae_o_ok:.2f} occ pts")
print(f"Shuffled-Target Ridge (Val):  MAE = {mae_t_shuf:.2f} tickets | {mae_o_shuf:.2f} occ pts")
print(f"No-Movie-Lags Ridge (Val):    MAE = {mae_t_nolag:.2f} tickets | {mae_o_nolag:.2f} occ pts")

Train <= 2025-01-12 (15,108) | Val <= 2025-04-03 (3,239) | Test > 2025-04-03 (3,237)
Normal Ridge (Val):           MAE = 14.56 tickets | 8.54 occ pts
Shuffled-Target Ridge (Val):  MAE = 29.92 tickets | 17.37 occ pts
No-Movie-Lags Ridge (Val):    MAE = 15.08 tickets | 8.81 occ pts
